# Hirriririir Multimodal Thigh Segmentation — Lambda Cloud

Runs the SegResNetDS model from [Hirriririir/Multimodal-Multiethnic-Thigh-Muscle-MRI-analysis](https://github.com/Hirriririir/Multimodal-Multiethnic-Thigh-Muscle-MRI-analysis) on fat-fraction NIfTI stacks.

**Designed for a Lambda Cloud GPU instance (A10 or similar).**

## Steps
1. Launch a Lambda instance and open JupyterLab
2. Upload this notebook to `~`
3. Ensure fat-fraction stacks are at `~/myosegmenTUM/myosegmenTUM/myosegmenTUM/FATFRACTION_collected/`
4. Run all cells top to bottom
5. The pretrained checkpoint (~330 MB) downloads automatically on first run
6. Results are saved to `~/multimodal_thigh_segs/`
7. Download results: `rsync -avz ubuntu@<IP>:~/multimodal_thigh_segs/ ./eval_notebooks/multimodal_thigh_segs/`
8. **Terminate the instance when done**

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'monai', 'SimpleITK'])
print('Dependencies ready')

In [ ]:
import glob, os
import numpy as np
import torch
import SimpleITK as sitk
from monai.networks.nets import SegResNetDS
from monai.inferers import sliding_window_inference

FATFRAC_DIR = os.path.expanduser(
    '~/myosegmenTUM/myosegmenTUM/myosegmenTUM/FATFRACTION_collected')
OUTPUT_DIR  = os.path.expanduser('~/multimodal_thigh_segs_fatfrac_v2')
CHECKPOINT  = os.path.expanduser('~/pretrained_segmentation_muscle.pt')

IMAGE_GLOB = os.path.join(FATFRAC_DIR, '*_FATFRACTION', '*_FATFRACTION_stack*.nii')

TARGET_SPACING = (0.7813, 0.7813, 4.0)
ROI_SIZE       = [336, 336, 88]
DEVICE         = 'cuda' if torch.cuda.is_available() else 'cpu'

LABEL_MAP = {
    1:  'Sartorius',
    2:  'Rectus_Femoris',
    3:  'Vastus_Lateralis',
    4:  'Vastus_Intermedius',
    5:  'Vastus_Medialis',
    6:  'Adductor_Magnus',
    7:  'Gracilis',
    8:  'Biceps_Femoris_Long',
    9:  'Semitendinosus',
    10: 'Semimembranosus',
    11: 'Biceps_Femoris_Short',
}

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Device:            ', DEVICE)
print('FATFRAC_DIR exists:', os.path.isdir(FATFRAC_DIR))
image_files = sorted(glob.glob(IMAGE_GLOB))
print(f'Found {len(image_files)} fat-fraction stacks')

In [ ]:
# Download pretrained checkpoint from GitHub releases (~330 MB)
if not os.path.exists(CHECKPOINT):
    import urllib.request
    print('Downloading checkpoint...')
    url = ('https://github.com/Hirriririir/Multimodal-Multiethnic-Thigh-Muscle-MRI-analysis'
           '/releases/download/1.0/pretrained_segmentation_muscle.pt')
    urllib.request.urlretrieve(url, CHECKPOINT)
    print(f'Done ({os.path.getsize(CHECKPOINT) // 1_000_000} MB)')
else:
    print('Checkpoint already present')

In [ ]:
model = SegResNetDS(
    spatial_dims=3,
    in_channels=1,
    out_channels=12,
    init_filters=32,
    blocks_down=(1, 2, 2, 4, 4),
    dsdepth=4,
    norm='INSTANCE',
    resolution=TARGET_SPACING,
)
ckpt  = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
state = (ckpt.get('state_dict') or ckpt.get('network_weights') or ckpt
         if isinstance(ckpt, dict) else ckpt)
missing, unexpected = model.load_state_dict(state, strict=False)
if missing:    print('Missing:   ', missing[:5])
if unexpected: print('Unexpected:', unexpected[:5])
model = model.to(DEVICE).eval()
print('Model ready on', DEVICE)

In [ ]:
def resample_sitk(sitk_img, new_spacing, interpolator=sitk.sitkLinear):
    orig_spacing = sitk_img.GetSpacing()
    orig_size    = sitk_img.GetSize()
    new_size = [
        int(round(orig_size[i] * orig_spacing[i] / new_spacing[i]))
        for i in range(3)
    ]
    r = sitk.ResampleImageFilter()
    r.SetOutputSpacing(new_spacing)
    r.SetSize(new_size)
    r.SetOutputDirection(sitk_img.GetDirection())
    r.SetOutputOrigin(sitk_img.GetOrigin())
    r.SetTransform(sitk.Transform())
    r.SetDefaultPixelValue(0)
    r.SetInterpolator(interpolator)
    return r.Execute(sitk_img)

def preprocess(nii_path):
    img  = sitk.ReadImage(nii_path, sitk.sitkFloat32)
    res  = resample_sitk(img, TARGET_SPACING)
    # SimpleITK returns (nz, ny, nx); MONAI/nibabel convention is (nx, ny, nz).
    arr  = sitk.GetArrayFromImage(res).astype(np.float32).transpose(2, 1, 0)
    mask = arr > 0
    if mask.any():
        arr[mask] = (arr[mask] - arr[mask].mean()) / (arr[mask].std() + 1e-8)
    return arr, res, img

def infer_volume(arr):
    t = torch.tensor(arr[None, None]).float().to(DEVICE)
    with torch.no_grad():
        out = sliding_window_inference(
            t, roi_size=ROI_SIZE, sw_batch_size=1,
            predictor=model, overlap=0.5, mode='gaussian'
        )
    logits = out[0] if isinstance(out, (list, tuple)) else out
    return torch.argmax(logits, dim=1).squeeze(0).cpu().numpy().astype(np.uint8)

In [ ]:
for nii_path in image_files:
    stem     = os.path.splitext(os.path.basename(nii_path))[0]
    out_path = os.path.join(OUTPUT_DIR, f'{stem}_thigh_seg.nii.gz')

    if os.path.exists(out_path):
        print(f'Skipping (done): {stem}')
        continue

    print(f'\nProcessing: {stem}')
    arr, res_ref, orig = preprocess(nii_path)
    print(f'  Array shape (nx, ny, nz): {arr.shape}')

    pred = infer_volume(arr)
    print(f'  Labels present: {sorted(np.unique(pred).tolist())}')

    # Transpose back from MONAI (nx, ny, nz) to SimpleITK (nz, ny, nx)
    pred_sitk = sitk.GetImageFromArray(pred.transpose(2, 1, 0))
    pred_sitk.CopyInformation(res_ref)
    pred_orig = sitk.Resample(pred_sitk, orig,
                               sitk.Transform(), sitk.sitkNearestNeighbor, 0)
    sitk.WriteImage(pred_orig, out_path)
    print(f'  Saved -> {out_path}')

    pred_arr = sitk.GetArrayFromImage(pred_orig)
    print(f'  {"Label":<6} {"Muscle":<25} {"Voxels":>10}')
    print(f'  {"-"*45}')
    for idx, name in LABEL_MAP.items():
        n = int((pred_arr == idx).sum())
        if n > 0:
            print(f'  {idx:<6} {name:<25} {n:>10,}')

print('\nAll done.')